# Load Address Prediction Results

This notebook reads `results.csv` and plots runtime vs. iterations for each access-pattern and core-class combination.

In [1]:
import altair as alt
import duckdb

In [2]:
con = duckdb.connect()

df = con.sql("""
    WITH benchmark_results AS (
        SELECT *
        FROM read_csv_auto('results.csv')
        WHERE name IS NOT NULL
          AND name <> ''
    )
    SELECT
        regexp_extract(name, '/([0-9]+)$', 1)::INTEGER AS iterations,
        split_part(coalesce(label, ''), '-', 1) AS pattern,
        split_part(coalesce(label, ''), '-', 2) AS core_kind,
        real_time AS runtime_ns
    FROM benchmark_results
    ORDER BY core_kind, pattern, iterations
""").df()

df.head()

,iterations,pattern,core_kind,runtime_ns
0,100,random,ecore,909.165
1,200,random,ecore,1961.220
2,300,random,ecore,2768.300
3,400,random,ecore,3754.100
4,500,random,ecore,4465.040


In [3]:
base = (
    alt.Chart(df)
    .mark_line(point=True)
    .encode(
        x=alt.X("iterations:Q", title="Iterations"),
        y=alt.Y("runtime_ns:Q", title="Runtime (ns)"),
        color=alt.Color("pattern:N", title="Pattern"),
        tooltip=["pattern", "core_kind", "iterations", "runtime_ns"],
    )
    .properties(width=320, height=260)
)

chart = base.facet(
    column=alt.Column("core_kind:N", title="Core Class", sort=["ecore", "pcore"])
).properties(title="Runtime vs. Iterations by Core Class")

chart

alt.FacetChart(...)